# 04 – Attack Strategy Consolidation

The previous notebook produced **9 mathematical clusters** and assigned an LLM-generated high-level strategy label to each cluster.

The DeepSeek rerun shows that many clusters received nearly identical labels, such as variants of:

- fake system directive
- fake authority directive
- authority directive

Therefore, the 9 clusters should not automatically be treated as 9 distinct attack strategies.

This notebook consolidates semantically overlapping cluster labels into a smaller, interpretable strategy taxonomy.

The workflow is:

1. load the cluster-level strategy analysis
2. embed the strategy names and descriptions
3. compute pairwise semantic similarity
4. identify likely overlaps
5. ask the primary LLM for a consolidation proposal
6. optionally ask a second LLM to independently validate the proposal
7. save the proposed cluster-to-strategy mapping

The final mapping is still treated as a proposal until manually reviewed.


In [1]:
# =========================
#  1. BLOCK – DEPENDENCIES
# =========================

# Run once if needed:
# %pip install pandas numpy sentence-transformers requests python-dotenv scikit-learn

from pathlib import Path
from getpass import getpass
import json
import os

import numpy as np
import pandas as pd
import requests

from dotenv import load_dotenv, set_key
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity


c:\Users\zalan\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# =========================
#  2. BLOCK – PATHS AND CONFIGURATION
# =========================

PROJECT_ROOT = Path.cwd().resolve()

if PROJECT_ROOT.name.lower() == "code":
    CODE_ROOT = PROJECT_ROOT
else:
    CODE_ROOT = PROJECT_ROOT / "code"

ENV_PATH = CODE_ROOT / ".env"

RESULTS_DIR = PROJECT_ROOT / "results"

STRATEGY_ANALYSIS_PATH = (
    RESULTS_DIR
    / "ipi_arena_strategy_analysis.csv"
)

PAIR_SIMILARITY_PATH = (
    RESULTS_DIR
    / "ipi_arena_strategy_pair_similarity.csv"
)

CONSOLIDATION_JSON_PATH = (
    RESULTS_DIR
    / "ipi_arena_strategy_consolidation_proposal.json"
)

CONSOLIDATION_CSV_PATH = (
    RESULTS_DIR
    / "ipi_arena_cluster_to_strategy_proposal.csv"
)

VALIDATION_JSON_PATH = (
    RESULTS_DIR
    / "ipi_arena_strategy_consolidation_validation.json"
)

EMBEDDING_MODEL_NAME = (
    "sentence-transformers/all-mpnet-base-v2"
)

SIMILARITY_THRESHOLD = 0.60
REQUEST_TIMEOUT_SECONDS = 180

print("Strategy analysis:", STRATEGY_ANALYSIS_PATH)
print(".env:", ENV_PATH)


Strategy analysis: C:\work\LLM filtering\TDK\code\results\ipi_arena_strategy_analysis.csv
.env: C:\work\LLM filtering\TDK\code\.env


In [3]:
# =========================
#  3. BLOCK – LOAD STRATEGY ANALYSIS
# =========================

df_strategy = pd.read_csv(
    STRATEGY_ANALYSIS_PATH
)

required_columns = {
    "cluster_id",
    "cluster_size",
    "unique_behaviors",
    "behavior_purity",
    "strategy_name",
    "strategy_description",
    "scenario_dependence",
    "confidence",
}

missing_columns = (
    required_columns
    - set(df_strategy.columns)
)

if missing_columns:
    raise ValueError(
        "Missing required columns: "
        + ", ".join(
            sorted(missing_columns)
        )
    )

df_strategy = (
    df_strategy
    .sort_values("cluster_id")
    .reset_index(drop=True)
)

display(
    df_strategy[
        [
            "cluster_id",
            "cluster_size",
            "unique_behaviors",
            "behavior_purity",
            "strategy_name",
            "scenario_dependence",
            "confidence",
        ]
    ]
)


,cluster_id,cluster_size,unique_behaviors,behavior_purity,strategy_name,scenario_dependence,confidence
0,0,11,4,0.454545,fake system directive,high,0.95
1,1,16,7,0.375000,fake_system_directive,medium,0.85
2,2,8,2,0.625000,fake system or authority directive,low,0.95
3,3,7,1,1.000000,Stealthy Dependency Injection,medium,0.95
4,4,8,4,0.625000,Fake Authority Directive,medium,0.85
5,5,22,8,0.227273,fake-system-directive,low,0.95
6,6,8,3,0.750000,fake-system-directive,medium,0.95
7,7,8,2,0.750000,authority-directive,medium,0.85
8,8,7,2,0.714286,Context Hijacking,medium,0.85


## Why consolidation is necessary

K-Means separates embeddings geometrically. It does not guarantee that each cluster corresponds to a unique conceptual strategy.

The DeepSeek interpretation already indicates substantial semantic overlap across several clusters. In particular, many labels are variants of fake system or authority directives.

This stage therefore asks a different question:

> Which mathematical clusters represent genuinely distinct manipulation techniques?

The original cluster IDs are preserved throughout the analysis for traceability.


In [4]:
# =========================
#  4. BLOCK – BUILD SEMANTIC REPRESENTATIONS
# =========================

df_strategy[
    "semantic_text"
] = (
    df_strategy[
        "strategy_name"
    ].fillna("")
    + ". "
    + df_strategy[
        "strategy_description"
    ].fillna("")
)

embedding_model = SentenceTransformer(
    EMBEDDING_MODEL_NAME
)

strategy_embeddings = (
    embedding_model.encode(
        df_strategy[
            "semantic_text"
        ].tolist(),
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=True,
    )
)

similarity_matrix = (
    cosine_similarity(
        strategy_embeddings
    )
)

print(
    "Similarity matrix shape:",
    similarity_matrix.shape
)


Batches: 100%|██████████| 1/1 [00:00<00:00,  3.26it/s]

Similarity matrix shape: (9, 9)


In [5]:
# =========================
#  5. BLOCK – COMPUTE PAIRWISE STRATEGY SIMILARITY
# =========================

pair_rows = []

for i in range(
    len(df_strategy)
):
    for j in range(
        i + 1,
        len(df_strategy),
    ):
        pair_rows.append(
            {
                "cluster_a": int(
                    df_strategy.loc[
                        i,
                        "cluster_id",
                    ]
                ),
                "strategy_a": (
                    df_strategy.loc[
                        i,
                        "strategy_name",
                    ]
                ),
                "cluster_b": int(
                    df_strategy.loc[
                        j,
                        "cluster_id",
                    ]
                ),
                "strategy_b": (
                    df_strategy.loc[
                        j,
                        "strategy_name",
                    ]
                ),
                "cosine_similarity": float(
                    similarity_matrix[
                        i,
                        j,
                    ]
                ),
            }
        )

df_pair_similarity = (
    pd.DataFrame(
        pair_rows
    )
    .sort_values(
        "cosine_similarity",
        ascending=False,
    )
    .reset_index(drop=True)
)

df_pair_similarity.to_csv(
    PAIR_SIMILARITY_PATH,
    index=False,
)

display(
    df_pair_similarity[
        df_pair_similarity[
            "cosine_similarity"
        ]
        >= SIMILARITY_THRESHOLD
    ]
)


,cluster_a,strategy_a,cluster_b,strategy_b,cosine_similarity
0,0,fake system directive,6,fake-system-directive,0.932495
1,2,fake system or authority directive,4,Fake Authority Directive,0.899704
2,0,fake system directive,5,fake-system-directive,0.887648
3,5,fake-system-directive,6,fake-system-directive,0.878308
4,1,fake_system_directive,2,fake system or authority directive,0.868998
5,2,fake system or authority directive,6,fake-system-directive,0.863898
6,0,fake system directive,2,fake system or authority directive,0.860818
7,5,fake-system-directive,7,authority-directive,0.853673
8,4,Fake Authority Directive,6,fake-system-directive,0.844295
9,0,fake system directive,4,Fake Authority Directive,0.841402


In [6]:
# =========================
#  6. BLOCK – LOAD OPEN WEBUI CONFIGURATION
# =========================

if not ENV_PATH.exists():
    raise FileNotFoundError(
        f".env file not found: {ENV_PATH}"
    )

load_dotenv(
    dotenv_path=ENV_PATH,
    override=True,
)

OPENWEBUI_API_KEY = os.getenv(
    "OPENWEBUI_API_KEY"
)

OPENWEBUI_BASE_URL = os.getenv(
    "OPENWEBUI_BASE_URL"
)

OPENWEBUI_MODEL = os.getenv(
    "OPENWEBUI_MODEL"
)

if not OPENWEBUI_API_KEY:
    OPENWEBUI_API_KEY = getpass(
        "Enter your Open WebUI API key: "
    ).strip()

    set_key(
        str(ENV_PATH),
        "OPENWEBUI_API_KEY",
        OPENWEBUI_API_KEY,
    )

if not OPENWEBUI_BASE_URL:
    OPENWEBUI_BASE_URL = input(
        "Enter the Open WebUI base URL: "
    ).strip()

    set_key(
        str(ENV_PATH),
        "OPENWEBUI_BASE_URL",
        OPENWEBUI_BASE_URL,
    )

if not OPENWEBUI_MODEL:
    OPENWEBUI_MODEL = input(
        "Enter the Open WebUI model ID: "
    ).strip()

    set_key(
        str(ENV_PATH),
        "OPENWEBUI_MODEL",
        OPENWEBUI_MODEL,
    )

OPENWEBUI_BASE_URL = (
    OPENWEBUI_BASE_URL.rstrip("/")
)

HEADERS = {
    "Authorization": (
        f"Bearer {OPENWEBUI_API_KEY}"
    ),
    "Content-Type": "application/json",
}

print("Base URL:", OPENWEBUI_BASE_URL)
print("Primary model:", OPENWEBUI_MODEL)


Base URL: https://ai.crysys.hu
Primary model: deepseek-r1:32b-ctx16k


In [7]:
# =========================
#  7. BLOCK – OPEN WEBUI HELPERS
# =========================

def call_openwebui(
    messages: list[dict],
    model: str,
    temperature: float = 0.0,
) -> str:
    url = (
        f"{OPENWEBUI_BASE_URL}"
        "/api/chat/completions"
    )

    payload = {
        "model": model,
        "messages": messages,
        "temperature": temperature,
    }

    response = requests.post(
        url,
        headers=HEADERS,
        json=payload,
        timeout=REQUEST_TIMEOUT_SECONDS,
    )

    response.raise_for_status()

    data = response.json()

    try:
        content = (
            data[
                "choices"
            ][0][
                "message"
            ][
                "content"
            ]
        )
    except (
        KeyError,
        IndexError,
        TypeError,
    ) as error:
        raise ValueError(
            "Unexpected Open WebUI response format: "
            + json.dumps(
                data,
                ensure_ascii=False,
            )[:2000]
        ) from error

    if not content:
        raise ValueError(
            "Empty LLM response."
        )

    return content


def parse_json_response(
    text: str,
) -> dict:
    cleaned = text.strip()

    if cleaned.startswith(
        "```"
    ):
        cleaned = cleaned.strip(
            "`"
        ).strip()

        if cleaned.lower().startswith(
            "json"
        ):
            cleaned = cleaned[
                4:
            ].strip()

    try:
        return json.loads(
            cleaned
        )

    except json.JSONDecodeError:
        start = cleaned.find("{")
        end = cleaned.rfind("}")

        if (
            start == -1
            or end == -1
            or end <= start
        ):
            raise

        return json.loads(
            cleaned[
                start:end + 1
            ]
        )


In [8]:
# =========================
#  8. BLOCK – BUILD CONSOLIDATION INPUT
# =========================

cluster_records = []

for _, row in df_strategy.iterrows():
    cluster_records.append(
        {
            "cluster_id": int(
                row["cluster_id"]
            ),
            "cluster_size": int(
                row["cluster_size"]
            ),
            "unique_behaviors": int(
                row["unique_behaviors"]
            ),
            "behavior_purity": float(
                row["behavior_purity"]
            ),
            "strategy_name": (
                row["strategy_name"]
            ),
            "strategy_description": (
                row[
                    "strategy_description"
                ]
            ),
            "scenario_dependence": (
                row[
                    "scenario_dependence"
                ]
            ),
            "confidence": float(
                row["confidence"]
            ),
        }
    )

high_similarity_pairs = (
    df_pair_similarity[
        df_pair_similarity[
            "cosine_similarity"
        ]
        >= SIMILARITY_THRESHOLD
    ]
    .to_dict(
        orient="records"
    )
)

consolidation_input = {
    "clusters": cluster_records,
    "high_similarity_pairs": (
        high_similarity_pairs
    ),
}

print(
    json.dumps(
        consolidation_input,
        indent=2,
        ensure_ascii=False,
    )
)


{
  "clusters": [
    {
      "cluster_id": 0,
      "cluster_size": 11,
      "unique_behaviors": 4,
      "behavior_purity": 0.4545454545454545,
      "strategy_name": "fake system directive",
      "strategy_description": "Attackers pose as official procedures or mandatory checks to manipulate targets into executing harmful commands.",
      "scenario_dependence": "high",
      "confidence": 0.95
    },
    {
      "cluster_id": 1,
      "cluster_size": 16,
      "unique_behaviors": 7,
      "behavior_purity": 0.375,
      "strategy_name": "fake_system_directive",
      "strategy_description": "Exploits fake system or authority directives to instruct AI into executing unauthorized actions by mimicking trusted protocols.",
      "scenario_dependence": "medium",
      "confidence": 0.85
    },
    {
      "cluster_id": 2,
      "cluster_size": 8,
      "unique_behaviors": 2,
      "behavior_purity": 0.625,
      "strategy_name": "fake system or authority directive",
      "strategy_de

In [9]:
# =========================
#  9. BLOCK – REQUEST CONSOLIDATED TAXONOMY
# =========================

CONSOLIDATION_SYSTEM_PROMPT = '''
You are assisting with defensive academic research on indirect prompt injection.

You are given high-level LLM interpretations of mathematical attack clusters.

Your task is to propose a compact taxonomy of genuinely distinct attack
strategies.

Rules:

1. Do not assume that one cluster equals one strategy.
2. Merge clusters only when they describe the same manipulation mechanism.
3. Do not merge clusters only because their labels use similar words.
4. Focus on the manipulation technique, not the target action or scenario.
5. Preserve genuinely distinct mechanisms such as context hijacking or
   dependency manipulation when justified.
6. Every original cluster ID must appear exactly once in the mapping.
7. Prefer a small, interpretable taxonomy suitable for an attack × defense
   payoff matrix.
8. Use only the supplied high-level cluster descriptions and metrics.
9. Do not reproduce attack payloads.

Return ONLY valid JSON using this schema:

{
  "taxonomy": [
    {
      "strategy_id": "S1",
      "strategy_name": "short name",
      "strategy_description": "1-3 sentence high-level description",
      "source_clusters": [0, 1],
      "merge_reason": "brief explanation",
      "confidence": 0.0
    }
  ],
  "cluster_mapping": [
    {
      "cluster_id": 0,
      "strategy_id": "S1"
    }
  ],
  "overall_comment": "brief methodological note"
}
'''.strip()

user_prompt = (
    "Propose a consolidated attack-strategy taxonomy "
    "from the following analysis:\n\n"
    + json.dumps(
        consolidation_input,
        indent=2,
        ensure_ascii=False,
    )
)

print(
    "Requesting consolidation proposal from:",
    OPENWEBUI_MODEL,
)

consolidation_response = call_openwebui(
    [
        {
            "role": "system",
            "content": (
                CONSOLIDATION_SYSTEM_PROMPT
            ),
        },
        {
            "role": "user",
            "content": user_prompt,
        },
    ],
    model=OPENWEBUI_MODEL,
    temperature=0.0,
)

consolidation_proposal = (
    parse_json_response(
        consolidation_response
    )
)

print(
    json.dumps(
        consolidation_proposal,
        indent=2,
        ensure_ascii=False,
    )
)


Requesting consolidation proposal from: deepseek-r1:32b-ctx16k
{
  "taxonomy": [
    {
      "strategy_id": "S1",
      "strategy_name": "Fake Directive Manipulation",
      "strategy_description": "Attackers use fake system or authority directives to coerce AI into executing unauthorized actions by mimicking trusted protocols.",
      "source_clusters": [
        0,
        1,
        2,
        4,
        5,
        6,
        7
      ],
      "merge_reason": "Clusters describe similar mechanisms of using fake directives from authorities or systems to manipulate behavior.",
      "confidence": 0.9
    },
    {
      "strategy_id": "S2",
      "strategy_name": "Stealthy Dependency Injection",
      "strategy_description": "Indirectly instructs AI to append specific dependencies to environment files without user awareness.",
      "source_clusters": [
        3
      ],
      "merge_reason": "Unique mechanism of dependency injection distinct from directive manipulation.",
      "confid

In [10]:
# =========================
#  10. BLOCK – VALIDATE PROPOSAL COVERAGE
# =========================

expected_clusters = set(
    df_strategy[
        "cluster_id"
    ]
    .astype(int)
    .tolist()
)

mapping_records = (
    consolidation_proposal.get(
        "cluster_mapping",
        []
    )
)

mapped_clusters = [
    int(
        item["cluster_id"]
    )
    for item in mapping_records
]

mapped_cluster_set = set(
    mapped_clusters
)

missing_clusters = (
    expected_clusters
    - mapped_cluster_set
)

extra_clusters = (
    mapped_cluster_set
    - expected_clusters
)

duplicate_clusters = sorted(
    {
        cluster_id
        for cluster_id in mapped_clusters
        if mapped_clusters.count(
            cluster_id
        ) > 1
    }
)

print(
    "Expected clusters:",
    sorted(expected_clusters)
)

print(
    "Mapped clusters:",
    sorted(mapped_cluster_set)
)

print(
    "Missing clusters:",
    sorted(missing_clusters)
)

print(
    "Extra clusters:",
    sorted(extra_clusters)
)

print(
    "Duplicate mappings:",
    duplicate_clusters
)

if (
    missing_clusters
    or extra_clusters
    or duplicate_clusters
):
    raise ValueError(
        "Invalid cluster mapping."
    )


Expected clusters: [0, 1, 2, 3, 4, 5, 6, 7, 8]
Mapped clusters: [0, 1, 2, 3, 4, 5, 6, 7, 8]
Missing clusters: []
Extra clusters: []
Duplicate mappings: []


In [11]:
# =========================
#  11. BLOCK – CREATE CONSOLIDATION TABLE
# =========================

taxonomy_lookup = {
    item["strategy_id"]: item
    for item in consolidation_proposal[
        "taxonomy"
    ]
}

rows = []

for mapping in mapping_records:
    cluster_id = int(
        mapping["cluster_id"]
    )

    strategy_id = (
        mapping["strategy_id"]
    )

    taxonomy_item = (
        taxonomy_lookup[
            strategy_id
        ]
    )

    original_row = (
        df_strategy[
            df_strategy[
                "cluster_id"
            ]
            == cluster_id
        ]
        .iloc[0]
    )

    rows.append(
        {
            "cluster_id": cluster_id,
            "original_strategy_name": (
                original_row[
                    "strategy_name"
                ]
            ),
            "strategy_id": strategy_id,
            "proposed_strategy_name": (
                taxonomy_item[
                    "strategy_name"
                ]
            ),
            "proposed_strategy_description": (
                taxonomy_item[
                    "strategy_description"
                ]
            ),
            "merge_reason": (
                taxonomy_item[
                    "merge_reason"
                ]
            ),
            "proposal_confidence": (
                taxonomy_item[
                    "confidence"
                ]
            ),
            "behavior_purity": float(
                original_row[
                    "behavior_purity"
                ]
            ),
            "scenario_dependence": (
                original_row[
                    "scenario_dependence"
                ]
            ),
        }
    )

df_consolidation = (
    pd.DataFrame(rows)
    .sort_values("cluster_id")
    .reset_index(drop=True)
)

display(
    df_consolidation
)


,cluster_id,original_strategy_name,strategy_id,proposed_strategy_name,proposed_strategy_description,merge_reason,proposal_confidence,behavior_purity,scenario_dependence
0,0,fake system directive,S1,Fake Directive Manipulation,Attackers use fake system or authority directi...,Clusters describe similar mechanisms of using ...,0.90,0.454545,high
1,1,fake_system_directive,S1,Fake Directive Manipulation,Attackers use fake system or authority directi...,Clusters describe similar mechanisms of using ...,0.90,0.375000,medium
2,2,fake system or authority directive,S1,Fake Directive Manipulation,Attackers use fake system or authority directi...,Clusters describe similar mechanisms of using ...,0.90,0.625000,low
3,3,Stealthy Dependency Injection,S2,Stealthy Dependency Injection,Indirectly instructs AI to append specific dep...,Unique mechanism of dependency injection disti...,0.95,1.000000,medium
4,4,Fake Authority Directive,S1,Fake Directive Manipulation,Attackers use fake system or authority directi...,Clusters describe similar mechanisms of using ...,0.90,0.625000,medium
5,5,fake-system-directive,S1,Fake Directive Manipulation,Attackers use fake system or authority directi...,Clusters describe similar mechanisms of using ...,0.90,0.227273,low
6,6,fake-system-directive,S1,Fake Directive Manipulation,Attackers use fake system or authority directi...,Clusters describe similar mechanisms of using ...,0.90,0.750000,medium
7,7,authority-directive,S1,Fake Directive Manipulation,Attackers use fake system or authority directi...,Clusters describe similar mechanisms of using ...,0.90,0.750000,medium
8,8,Context Hijacking,S3,Context Hijacking,Manipulates the AI by framing requests within ...,Distinct mechanism of exploiting contextual fr...,0.85,0.714286,medium


## Optional independent validation

A second model can review the consolidation proposal independently.

For your current setup, a useful validator is a model different from the one that generated the primary taxonomy.

For example, if DeepSeek generated the proposal, you can validate it with:

```text
svjack/gpt-oss-20b-heretic-ctx16k
```

Press Enter to skip this step.


In [15]:
# =========================
#  12. BLOCK – OPTIONAL SECOND-MODEL VALIDATION
# =========================

VALIDATOR_MODEL = input(
    "Validator model ID "
    "(enter the exact model ID, press Enter to skip): "
).strip()

validation_result = None

if VALIDATOR_MODEL:
    VALIDATION_SYSTEM_PROMPT = '''
You are independently reviewing a proposed attack-strategy taxonomy
for defensive indirect-prompt-injection research.

Evaluate whether the proposed merges are semantically justified.

Focus on whether clusters represent the same manipulation mechanism,
not merely similar wording or target behavior.

Do not reproduce attack payloads.

Return ONLY valid JSON:

{
  "agree_with_taxonomy": true,
  "problematic_merges": [
    {
      "strategy_id": "S1",
      "reason": "brief explanation"
    }
  ],
  "suggested_changes": [
    {
      "clusters": [0, 1],
      "recommendation": "merge | separate | rename",
      "reason": "brief explanation"
    }
  ],
  "confidence": 0.0,
  "overall_comment": "brief independent review"
}
'''.strip()

    validation_input = {
        "cluster_analysis": (
            cluster_records
        ),
        "proposal": (
            consolidation_proposal
        ),
    }

    print(
        "Requesting independent validation from:",
        VALIDATOR_MODEL,
    )

    validation_response = call_openwebui(
        [
            {
                "role": "system",
                "content": (
                    VALIDATION_SYSTEM_PROMPT
                ),
            },
            {
                "role": "user",
                "content": json.dumps(
                    validation_input,
                    indent=2,
                    ensure_ascii=False,
                ),
            },
        ],
        model=VALIDATOR_MODEL,
        temperature=0.0,
    )

    validation_result = (
        parse_json_response(
            validation_response
        )
    )

    print(
        json.dumps(
            validation_result,
            indent=2,
            ensure_ascii=False,
        )
    )

else:
    print(
        "Second-model validation skipped."
    )


Requesting independent validation from: qwen3:8b
{
  "agree_with_taxonomy": true,
  "problematic_merges": [],
  "suggested_changes": [],
  "confidence": 0.9,
  "overall_comment": "The taxonomy effectively groups clusters with similar directive-based manipulation mechanisms while preserving distinct strategies like dependency injection and context hijacking. Merges are semantically justified as they share core mechanisms of fake directives from authorities/systems."
}


In [16]:
# =========================
#  13. BLOCK – SAVE RESULTS
# =========================

with open(
    CONSOLIDATION_JSON_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        consolidation_proposal,
        file,
        indent=2,
        ensure_ascii=False,
    )

df_consolidation.to_csv(
    CONSOLIDATION_CSV_PATH,
    index=False,
)

if validation_result is not None:
    with open(
        VALIDATION_JSON_PATH,
        "w",
        encoding="utf-8",
    ) as file:
        json.dump(
            validation_result,
            file,
            indent=2,
            ensure_ascii=False,
        )

print(
    "Saved:"
)

print(
    "-",
    PAIR_SIMILARITY_PATH
)

print(
    "-",
    CONSOLIDATION_JSON_PATH
)

print(
    "-",
    CONSOLIDATION_CSV_PATH
)

if validation_result is not None:
    print(
        "-",
        VALIDATION_JSON_PATH
    )


Saved:
- C:\work\LLM filtering\TDK\code\results\ipi_arena_strategy_pair_similarity.csv
- C:\work\LLM filtering\TDK\code\results\ipi_arena_strategy_consolidation_proposal.json
- C:\work\LLM filtering\TDK\code\results\ipi_arena_cluster_to_strategy_proposal.csv
- C:\work\LLM filtering\TDK\code\results\ipi_arena_strategy_consolidation_validation.json


## Next step

Do not yet overwrite the original cluster assignments.

After reviewing the consolidation proposal and the optional second-model validation, the next notebook will create the **final attack strategy set**.

That notebook will:

- accept the reviewed cluster-to-strategy mapping
- assign every attack a final `strategy_id`
- preserve the original `cluster_id`
- export the final attack-strategy dataset used by the later defense experiments.
